# Modeling Klasifikasi: Naive Bayes vs KNN

Notebook ini melanjutkan `text-preprocessing.ipynb`. Pada notebook sebelumnya,
200 artikel detik.com (100 sport, 100 finance) sudah melalui pembersihan
teks, normalisasi slang, penghapusan stopword, stemming, hingga vektorisasi
TF-IDF menghasilkan matriks 200 dokumen x 4.970 fitur. Hasil itu disimpan ke
tiga berkas:

- `tfidf_sparse.npz` -- matriks TF-IDF dalam format sparse
- `tfidf_features.txt` -- daftar nama fitur (kata) sesuai urutan kolom
- `tfidf_docs.csv` -- pasangan `id` dan `label` (sport/finance) tiap dokumen

Ketiganya dimuat kembali di notebook ini. Tidak ada pengulangan tahap
pembersihan teks, tokenisasi, atau vektorisasi -- fokus notebook ini murni
pada tahap **modeling**, yaitu melatih model klasifikasi yang dapat
menebak kategori berita (sport atau finance) hanya dari representasi
TF-IDF-nya.

## Mengapa dua algoritma ini?

| Algoritma | Prinsip kerja | Alasan dipilih |
|---|---|---|
| **Naive Bayes (Multinomial)** | Menghitung peluang suatu dokumen masuk kelas tertentu berdasarkan frekuensi kemunculan tiap kata, dengan asumsi independensi antar kata (mengabaikan urutan/konteks) | Dirancang khusus untuk fitur berupa hitungan/frekuensi non-negatif seperti TF-IDF; ringan secara komputasi dan menjadi baseline klasik untuk klasifikasi teks |
| **K-Nearest Neighbors (KNN)** | Mengklasifikasi dokumen baru berdasarkan mayoritas label dari k dokumen paling mirip di data latih | Pendekatan berbasis kemiripan yang tidak mengasumsikan independensi antar fitur, sehingga menjadi pembanding yang berbeda filosofi dari Naive Bayes |

Kedua model dilatih pada data yang identik (pembagian latih/uji yang sama)
supaya perbandingan performa di akhir notebook ini adil, lalu model dengan
skor lebih baik dipilih sebagai model akhir.

## 1. Memuat Data Hasil Preprocessing

Sel di bawah membaca kembali ketiga berkas hasil preprocessing tanpa
memodifikasi isinya. `X` adalah matriks fitur TF-IDF (sparse, agar hemat
memori karena mayoritas selnya bernilai nol), sedangkan `y` adalah label
kelas yang diambil dari kolom `label` pada `tfidf_docs.csv`. Urutan baris
pada `X` dan `y` harus konsisten -- keduanya memang berasal dari proses
yang sama sehingga urutannya sudah selaras.

In [42]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

X = sp.load_npz("tfidf_sparse.npz")
feature_names = pd.read_csv("tfidf_features.txt", header=None)[0].to_numpy()
docs = pd.read_csv("tfidf_docs.csv")
y = docs["label"].to_numpy()

pd.DataFrame({
    "metrik": ["dokumen", "fitur", "kelas"],
    "nilai": [X.shape[0], X.shape[1], ", ".join(sorted(set(y)))],
})

,metrik,nilai
0,dokumen,200
1,fitur,4970
2,kelas,"finance, sport"


## 2. Split Data Latih dan Uji

Data dibagi 80:20 -- 80% untuk melatih model, 20% disisihkan sebagai data
uji yang sama sekali tidak dilihat model selama pelatihan. Pembagian ini
memakai **stratifikasi** pada kolom label (`stratify=y`), artinya proporsi
sport dan finance dijaga tetap seimbang di kedua himpunan, bukan dibagi
secara acak murni yang berisiko membuat satu himpunan didominasi satu
kelas saja. `random_state=42` dipakai supaya hasil pembagian dapat
direproduksi persis jika notebook dijalankan ulang.

In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

pd.DataFrame({
    "himpunan": ["latih", "uji"],
    "jumlah": [X_train.shape[0], X_test.shape[0]],
    "sport": [(y_train == "sport").sum(), (y_test == "sport").sum()],
    "finance": [(y_train == "finance").sum(), (y_test == "finance").sum()],
})

,himpunan,jumlah,sport,finance
0,latih,160,80,80
1,uji,40,20,20


## 3. Model 1: Naive Bayes (Multinomial)

`MultinomialNB` dari scikit-learn dipakai karena varian ini secara khusus
memodelkan fitur berupa frekuensi kata (count atau TF-IDF), berbeda dengan
`GaussianNB` yang mengasumsikan fitur berdistribusi normal kontinu.
Parameter smoothing `alpha=1.0` (Laplace smoothing) dipakai sebagai
default -- ini mencegah probabilitas menjadi nol saat model bertemu kata
yang tidak muncul sama sekali pada satu kelas selama pelatihan.

Model dilatih pada `X_train`/`y_train`, lalu dipakai memprediksi label
`X_test`. Hasilnya dievaluasi dengan:

- **`classification_report`** -- menyajikan precision, recall, dan
  f1-score per kelas, sekaligus rata-rata makro (bobot sama untuk tiap
  kelas) dan tertimbang (weighted, mengikuti jumlah data tiap kelas).
- **Confusion matrix** -- tabel silang antara label aktual dan label hasil
  prediksi, memperlihatkan persis di mana model salah menebak (jika ada).

In [44]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

nb = MultinomialNB()
nb.fit(X_train, y_train)
pred_nb = nb.predict(X_test)

report_nb = pd.DataFrame(classification_report(y_test, pred_nb, output_dict=True)).T
report_nb

,precision,recall,f1-score,support
finance,1.0,1.0,1.0,20.0
sport,1.0,1.0,1.0,20.0
accuracy,1.0,1.0,1.0,1.0
macro avg,1.0,1.0,1.0,40.0
weighted avg,1.0,1.0,1.0,40.0


In [45]:
labels_order = sorted(set(y))

pd.DataFrame(
    confusion_matrix(y_test, pred_nb, labels=labels_order),
    index=[f"aktual_{l}" for l in labels_order],
    columns=[f"prediksi_{l}" for l in labels_order],
)

,prediksi_finance,prediksi_sport
aktual_finance,20,0
aktual_sport,0,20


**Cara membaca confusion matrix di atas:** baris menunjukkan label
sebenarnya, kolom menunjukkan label yang ditebak model. Angka pada
diagonal (`aktual_finance`-`prediksi_finance` dan `aktual_sport`-`prediksi_sport`)
adalah prediksi yang benar; angka di luar diagonal adalah kesalahan --
misalnya nilai pada `aktual_sport`-`prediksi_finance` berarti dokumen
sport yang keliru ditebak sebagai finance.

## 4. Model 2: K-Nearest Neighbors (KNN)

KNN tidak "belajar" pola secara eksplisit seperti Naive Bayes -- ia
menyimpan seluruh data latih, dan saat memprediksi dokumen baru, ia
mencari k dokumen latih paling mirip lalu mengambil label mayoritas di
antara mereka.

Dua keputusan desain penting untuk kasus ini:

1. **Metrik jarak `cosine`, bukan `euclidean`.** Vektor TF-IDF berdimensi
   sangat tinggi (4.970 fitur) dan sangat *sparse* (mayoritas nol).
   Jarak euclidean pada ruang seperti ini mudah bias oleh panjang dokumen,
   sedangkan cosine similarity mengukur kemiripan arah/proporsi kata,
   sehingga lebih umum dipakai untuk data teks.
2. **Pemilihan k lewat pemindaian (scan), bukan ditebak sekali.** Nilai k
   yang terlalu kecil membuat model sensitif terhadap noise (overfitting
   ke tetangga terdekat), sedangkan k terlalu besar membuat batas antar
   kelas menjadi kabur (underfitting). Beberapa nilai k dicoba (3, 5, 7,
   9 -- semuanya ganjil agar tidak ada suara seri pada klasifikasi biner)
   dan performanya dibandingkan pada data uji.

Catatan: karena ukuran data hanya 200 dokumen, pemilihan k di sini memakai
evaluasi langsung pada data uji sebagai perbandingan awal, bukan
cross-validation berlapis yang idealnya dipakai pada dataset lebih besar.

In [46]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

knn_scan = pd.DataFrame([
    {
        "k": k,
        "akurasi": accuracy_score(
            y_test,
            KNeighborsClassifier(n_neighbors=k, metric="cosine")
                .fit(X_train, y_train).predict(X_test)
        ),
        "f1_macro": f1_score(
            y_test,
            KNeighborsClassifier(n_neighbors=k, metric="cosine")
                .fit(X_train, y_train).predict(X_test),
            average="macro"
        ),
    }
    for k in [3, 5, 7, 9]
])
knn_scan

,k,akurasi,f1_macro
0,3,0.975,0.974984
1,5,0.975,0.974984
2,7,0.975,0.974984
3,9,0.975,0.974984


Nilai k dengan f1-score makro tertinggi pada tabel di atas dipilih
sebagai konfigurasi final. Jika beberapa k menghasilkan skor yang sama
persis, k terkecil dipilih (mengikuti urutan tabel) karena model lebih
sederhana dan lebih murah secara komputasi saat dipakai memprediksi data
baru nantinya.

In [47]:
best_k = int(knn_scan.loc[knn_scan["f1_macro"].idxmax(), "k"])

knn = KNeighborsClassifier(n_neighbors=best_k, metric="cosine")
knn.fit(X_train, y_train)
pred_knn = knn.predict(X_test)

report_knn = pd.DataFrame(classification_report(y_test, pred_knn, output_dict=True)).T
report_knn

,precision,recall,f1-score,support
finance,0.952381,1.000,0.975610,20.000
sport,1.000000,0.950,0.974359,20.000
accuracy,0.975000,0.975,0.975000,0.975
macro avg,0.976190,0.975,0.974984,40.000
weighted avg,0.976190,0.975,0.974984,40.000


In [48]:
pd.DataFrame(
    confusion_matrix(y_test, pred_knn, labels=labels_order),
    index=[f"aktual_{l}" for l in labels_order],
    columns=[f"prediksi_{l}" for l in labels_order],
)

,prediksi_finance,prediksi_sport
aktual_finance,20,0
aktual_sport,1,19


## 5. Perbandingan Model

Kedua model sudah dievaluasi pada himpunan uji yang **identik**, sehingga
angka pada tabel berikut bisa dibandingkan langsung secara adil. Empat
metrik dipakai:

- **Akurasi** - proporsi seluruh prediksi yang benar dari total data uji.
  Metrik ini cukup diandalkan di sini karena kelas seimbang (20 sport,
  20 finance pada data uji).
- **Presisi makro** - dari semua dokumen yang ditebak masuk suatu kelas,
  berapa persen yang benar-benar kelas itu; dirata-rata sama bobot untuk
  kedua kelas.
- **Recall makro** - dari semua dokumen yang sebenarnya milik suatu
  kelas, berapa persen yang berhasil ditebak benar; juga dirata-rata sama
  bobot.
- **F1-score makro** - rata-rata harmonik presisi dan recall, dipakai
  sebagai kriteria utama pemilihan model karena mempertimbangkan kedua
  sisi kesalahan (false positive maupun false negative) sekaligus.

In [49]:
comparison = pd.DataFrame({
    "model": ["Naive Bayes", f"KNN (k={best_k})"],
    "akurasi": [accuracy_score(y_test, pred_nb), accuracy_score(y_test, pred_knn)],
    "presisi_macro": [report_nb.loc["macro avg", "precision"], report_knn.loc["macro avg", "precision"]],
    "recall_macro": [report_nb.loc["macro avg", "recall"], report_knn.loc["macro avg", "recall"]],
    "f1_macro": [report_nb.loc["macro avg", "f1-score"], report_knn.loc["macro avg", "f1-score"]],
})
comparison

,model,akurasi,presisi_macro,recall_macro,f1_macro
0,Naive Bayes,1.000,1.00000,1.000,1.000000
1,KNN (k=3),0.975,0.97619,0.975,0.974984


## 6. Kesimpulan

Sel berikut menyusun kesimpulan secara otomatis berdasarkan angka pada
tabel perbandingan di atas -- model dengan f1-score makro tertinggi
dipilih sebagai model akhir. Pendekatan ini dipakai (bukan menuliskan nama
model secara manual) supaya kesimpulan selalu konsisten dengan angka hasil
run, termasuk jika data atau parameter di sel-sel sebelumnya diubah di
kemudian hari.

In [50]:
from IPython.display import Markdown, display

best_idx = comparison["f1_macro"].idxmax()
best_model = comparison.loc[best_idx, "model"]
best_acc = comparison.loc[best_idx, "akurasi"]
best_f1 = comparison.loc[best_idx, "f1_macro"]

display(Markdown(f"""### Model terpilih: {best_model}

Pada data uji (20% dari 200 dokumen, seimbang antara sport dan finance),
model **{best_model}** memberikan hasil terbaik dengan akurasi
{best_acc:.2%} dan f1-score makro {best_f1:.2%}, mengungguli model
pembanding pada tabel perbandingan di atas. Model ini yang dipilih untuk
tahap klasifikasi kategori berita selanjutnya.

Perlu dicatat, skor setinggi ini wajar untuk dataset kecil (200 dokumen)
dengan dua topik yang kosakatanya cukup berbeda (istilah olahraga vs
istilah keuangan jarang tumpang tindih). Performa pada data baru di luar
korpus ini -- terutama artikel yang membahas kedua topik sekaligus, misal
berita bisnis klub sepak bola -- perlu diuji ulang sebelum model dipakai
secara produksi."""))

### Model terpilih: Naive Bayes

Pada data uji (20% dari 200 dokumen, seimbang antara sport dan finance),
model **Naive Bayes** memberikan hasil terbaik dengan akurasi
100.00% dan f1-score makro 100.00%, mengungguli model
pembanding pada tabel perbandingan di atas. Model ini yang dipilih untuk
tahap klasifikasi kategori berita selanjutnya.

Perlu dicatat, skor setinggi ini wajar untuk dataset kecil (200 dokumen)
dengan dua topik yang kosakatanya cukup berbeda (istilah olahraga vs
istilah keuangan jarang tumpang tindih). Performa pada data baru di luar
korpus ini -- terutama artikel yang membahas kedua topik sekaligus, misal
berita bisnis klub sepak bola -- perlu diuji ulang sebelum model dipakai
secara produksi.